## Persistent Landing - Delta Lake (Non-Structured)

## Execution Notes and Batch Logging
This notebook builds a Delta file catalogue for JSON and image assets. The JSON and image processors log each S3 paginator page as a batch, each object/file processed, records discovered, and catalogue rows appended.


Even though Delta tables are typically used to store structured data, they can also be used to store metadata extracted from unstructured or semi-structured data.

**Landing file catalogue governance note.** JSON and image catalogue rows include source, validation, schema, owner, steward, classification, PII, and retention fields. This mirrors the DAG catalogue builder locally and does not import DAG helpers.


**Importing Useful Libraries**

In [1]:
import os
import boto3
import duckdb
import hashlib
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl
from PIL import Image
from PIL.ExifTags import TAGS
import pandas as pd
import json
import io

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "writer"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")


In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [3]:
def get_deep_keys(data,level=0):
    # If it's a list, dive into the first element
    if isinstance(data, list) and len(data) > 0:
        return get_deep_keys(data[0],level+1)

    # If it's finally a dictionary, return the keys
    if isinstance(data, dict):
        return list(data.keys()),level

    # If it's a primitive (like a string or number) or empty
    return [],level

def extract_timestamp_from_filename(filename):
    # Strip extension and split by underscore
    name_part = os.path.splitext(filename)[0]
    raw_ts = name_part.split('_')[-1]

    try:
        # Convert string epoch to a readable datetime object
        dt_object = datetime.fromtimestamp(int(raw_ts))
        return dt_object
    except (ValueError, IndexError):
        # Fallback if the filename doesn't follow the pattern
        return datetime.now()
        
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}
LANDING_POLICY_DEFAULTS = {
    "owner": "data_engineering_team",
    "data_steward": "bdm_project_team",
    "pii_flag": "no_direct_pii",
    "retention_policy": "course_project_retained_until_assessment_archive",
}

CATALOG_STRING_COLUMNS = [
    "file_id",
    "file_path",
    "source_type",
    "file_type",
    "event_time",
    "metadata_blob",
    "source_system",
    "ingestion_time",
    "source_file_path",
    "validation_status",
    "schema_version",
    "owner",
    "data_steward",
    "data_classification",
    "pii_flag",
    "retention_policy",
]


def normalize_catalog_df(df: pd.DataFrame) -> pd.DataFrame:
    # Keep Delta catalog writes stable even when a batch has all-null values.
    normalized = df.copy()
    for column in CATALOG_STRING_COLUMNS:
        if column not in normalized.columns:
            normalized[column] = ""
        normalized[column] = normalized[column].fillna("").astype(str)

    for column, default_value in LANDING_POLICY_DEFAULTS.items():
        normalized[column] = normalized[column].replace("", default_value).fillna(default_value).astype(str)

    normalized["validation_status"] = normalized["validation_status"].replace("", "valid")
    normalized["schema_version"] = normalized["schema_version"].replace("", "landing_file_catalog_v1")

    if "record_count" not in normalized.columns:
        normalized["record_count"] = 0
    normalized["record_count"] = normalized["record_count"].fillna(0).astype("int64")

    if "processed_at" not in normalized.columns:
        normalized["processed_at"] = pd.Timestamp.now()
    normalized["processed_at"] = pd.to_datetime(normalized["processed_at"], errors="coerce").fillna(pd.Timestamp.now())

    return normalized[[
        "file_id",
        "source_type",
        "file_path",
        "file_type",
        "event_time",
        "record_count",
        "metadata_blob",
        "processed_at",
        "source_system",
        "ingestion_time",
        "source_file_path",
        "validation_status",
        "schema_version",
        "owner",
        "data_steward",
        "data_classification",
        "pii_flag",
        "retention_policy",
    ]]


**Semistructured Data**

In [11]:
def process_json(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")
    processed_files = 0

    for batch_no, page in enumerate(paginator.paginate(Bucket=bucket, Prefix=prefix), start=1):
        page_objects = [
            obj for obj in page.get("Contents", [])
            if not (obj["Key"].endswith("/") or obj['Size'] == 0)
        ]
        print(f"[json catalogue batch {batch_no}] candidates={len(page_objects)} prefix=s3://{bucket}/{prefix}")

        for object_no, obj in enumerate(page_objects, start=1):
            src_key = obj["Key"]

            # 2. Read and Parse
            response = s3.get_object(Bucket=bucket, Key=src_key)
            object_metadata = response.get('Metadata', {})
            content = response['Body'].read().decode('utf-8')
            data = json.loads(content)

            # 3. Extract Deep Metadata
            keys_list, level = get_deep_keys(data)

            # 4. Correct Record Counting (Flattening for the count)
            # This ensures [[{}, {}]] returns 2, not 1
            temp_data = data
            for _ in range(level):
                if isinstance(temp_data, list) and len(temp_data) > 0:
                    temp_data = [item for sublist in temp_data for item in (sublist if isinstance(sublist, list) else [sublist])]
            record_count = len(temp_data)

            # 5. Build the Metadata Blob (The "Table inside a Table")
            # This blob changes structure based on file type
            metadata_blob = {
                "nesting_level": level,
                "schema_keys": keys_list,
                "file_size_bytes": obj['Size']
            }

            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = pd.DataFrame([{
                "file_id": filename,
                "source_type": src_key.split('/')[2],
                "file_path": src_key,
                "file_type": "JSON",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": record_count,
                "metadata_blob": json.dumps(metadata_blob),
                "processed_at": pd.Timestamp.now(),
                "source_system": object_metadata.get("source_system", "kafka"),
                "ingestion_time": object_metadata.get("ingestion_time") or pd.Timestamp.now(tz="UTC").isoformat(),
                "source_file_path": object_metadata.get("source_file_path") or f"s3://{bucket}/{src_key}",
                "validation_status": object_metadata.get("validation_status", "valid"),
                "schema_version": object_metadata.get("schema_version", "landing_file_catalog_v1"),
                "owner": object_metadata.get("owner", LANDING_POLICY_DEFAULTS["owner"]),
                "data_steward": object_metadata.get("data_steward", LANDING_POLICY_DEFAULTS["data_steward"]),
                "data_classification": object_metadata.get("data_classification", "public_environmental_observation"),
                "pii_flag": object_metadata.get("pii_flag", LANDING_POLICY_DEFAULTS["pii_flag"]),
                "retention_policy": object_metadata.get("retention_policy", LANDING_POLICY_DEFAULTS["retention_policy"]),
            }])

            # 7. Append to Master Catalog
            metadata_row = normalize_catalog_df(metadata_row)
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_row,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )
            processed_files += 1
            print(f"[json catalogue batch {batch_no}.{object_no}] appended file_id={filename}, records={record_count}, schema_keys={len(keys_list)}")

    print(f"JSON catalogue ingestion complete. Files processed: {processed_files}")

In [5]:
process_json("landing-zone", "persistent-landing/semistructured/")


[json catalogue batch 1] candidates=2 prefix=s3://landing-zone/persistent-landing/semistructured/
[json catalogue batch 1.1] Processing: persistent-landing/semistructured/airquality-barcelona.json
[json catalogue batch 1.1] appended file_id=airquality-barcelona.json, records=57, schema_keys=17
[json catalogue batch 1.2] Processing: persistent-landing/semistructured/weather-barcelona.json
[json catalogue batch 1.2] appended file_id=weather-barcelona.json, records=19, schema_keys=7
JSON catalogue ingestion complete. Files processed: 2


In [6]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [7]:
# 1. Show all columns (don't hide the middle ones)
pd.set_option('display.max_columns', None)

# 2. Show the full content of each cell (don't truncate long JSON/strings)
pd.set_option('display.max_colwidth', None)

# 3. Show all rows (optional: only use if the table is small, e.g., < 100 rows)
# pd.set_option('display.max_rows', None)

# Execute and display
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,source_type,file_path,file_type,event_time,record_count,metadata_blob,processed_at,source_system,ingestion_time,source_file_path,validation_status,schema_version,owner,data_steward,data_classification,pii_flag,retention_policy
0,weather-barcelona.json,weather-barcelona.json,persistent-landing/semistructured/weather-barcelona.json,JSON,2026-06-06 16:51:58.851623,19,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 2641}",2026-06-06 16:51:58.851642,kafka,2026-06-06T16:51:29.897810+00:00,kafka://weather-barcelona/notebook-aggregation,valid,landing_raw_v1,data_engineering_team,bdm_project_team,public_environmental_observation,no_direct_pii,course_project_retained_until_assessment_archive
1,airquality-barcelona.json,airquality-barcelona.json,persistent-landing/semistructured/airquality-barcelona.json,JSON,2026-06-06 16:51:58.741503,57,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 104633}",2026-06-06 16:51:58.741567,kafka,2026-06-06T16:51:30.775270+00:00,kafka://airquality-barcelona/notebook-aggregation,valid,landing_raw_v1,data_engineering_team,bdm_project_team,public_environmental_observation,no_direct_pii,course_project_retained_until_assessment_archive


**Unstructured Data**

In [8]:
def process_image(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")
    processed_images = 0

    for batch_no, page in enumerate(paginator.paginate(Bucket=bucket, Prefix=prefix), start=1):
        page_records = []
        page_objects = [
            obj for obj in page.get("Contents", [])
            if not (obj["Key"].endswith("/") or obj['Size'] == 0)
        ]
        print(f"[image catalogue batch {batch_no}] candidates={len(page_objects)} prefix=s3://{bucket}/{prefix}")

        for object_no, obj in enumerate(page_objects, start=1):
            src_key = obj["Key"]
            metadata = {}
            metadata_blob = {}
            try:
                # 2. Read and Parse
                response = s3.get_object(Bucket=bucket, Key=src_key)
                content = response['Body'].read()

                img = Image.open(io.BytesIO(content))
                metadata = response.get('Metadata', {})
                width, height = img.size

                metadata_blob = {
                                    "label": metadata.get('label'),
                                    "url": metadata.get('url'),
                                    "file_size_bytes": obj['Size'],
                                    "content_type": response.get('ContentType'),
                                    "width": width,
                                    "height": height,
                                    "aspect_ratio": round(width / height, 2) if height > 0 else 0,
                                    "image_mode": img.mode,
                                    "is_corrupted": False,
                                    "md5": hashlib.md5(content).hexdigest() # for duplicate detection
                                }

            except Exception as e:
                print(f"Error parsing image {src_key}: {e}")
                metadata_blob.update({
                    "is_corrupted": True,
                    "error_msg": str(e),
                    "width": 0, "height": 0, "aspect_ratio": 0, "image_mode": "unknown"
                })



            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = {
                "file_id": filename,
                "file_path": src_key,
                "source_type": metadata.get('source') or src_key.split('/')[2],
                "file_type": "Image",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": 1, # For image data it alaways 1
                "metadata_blob": json.dumps(metadata_blob),
                "processed_at": pd.Timestamp.now(),
                "source_system": metadata.get("source_system") or metadata.get("source") or "landing-zone",
                "ingestion_time": metadata.get("ingestion_time") or pd.Timestamp.now(tz="UTC").isoformat(),
                "source_file_path": metadata.get("source_file_path") or f"s3://{bucket}/{src_key}",
                "validation_status": metadata.get("validation_status", "valid"),
                "schema_version": metadata.get("schema_version", "landing_file_catalog_v1"),
                "owner": metadata.get("owner", LANDING_POLICY_DEFAULTS["owner"]),
                "data_steward": metadata.get("data_steward", LANDING_POLICY_DEFAULTS["data_steward"]),
                "data_classification": metadata.get("data_classification", "public_image_metadata"),
                "pii_flag": metadata.get("pii_flag", LANDING_POLICY_DEFAULTS["pii_flag"]),
                "retention_policy": metadata.get("retention_policy", LANDING_POLICY_DEFAULTS["retention_policy"]),
            }

            page_records.append(metadata_row)

        # 7. Append to Master Catalog
        if page_records:
            metadata_df = normalize_catalog_df(pd.DataFrame(page_records))
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_df,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )
        processed_images += len(page_records)
        print(f"[image catalogue batch {batch_no}] uploaded: {len(page_records)} images; cumulative={processed_images}")

    print(f"Image catalogue ingestion complete. Images processed: {processed_images}")

In [9]:
process_image("landing-zone","persistent-landing/unstructured/image")

[image catalogue batch 1] candidates=999 prefix=s3://landing-zone/persistent-landing/unstructured/image
[image catalogue batch 1] uploaded: 999 images; cumulative=999
[image catalogue batch 2] candidates=1000 prefix=s3://landing-zone/persistent-landing/unstructured/image
[image catalogue batch 2] uploaded: 1000 images; cumulative=1999
[image catalogue batch 3] candidates=1000 prefix=s3://landing-zone/persistent-landing/unstructured/image
[image catalogue batch 3] uploaded: 1000 images; cumulative=2999
[image catalogue batch 4] candidates=1000 prefix=s3://landing-zone/persistent-landing/unstructured/image
[image catalogue batch 4] uploaded: 1000 images; cumulative=3999
[image catalogue batch 5] candidates=1000 prefix=s3://landing-zone/persistent-landing/unstructured/image
[image catalogue batch 5] uploaded: 1000 images; cumulative=4999
[image catalogue batch 6] candidates=1000 prefix=s3://landing-zone/persistent-landing/unstructured/image
[image catalogue batch 6] uploaded: 1000 images;

In [10]:
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,source_type,file_path,file_type,event_time,record_count,metadata_blob,processed_at,source_system,ingestion_time,source_file_path,validation_status,schema_version,owner,data_steward,data_classification,pii_flag,retention_policy
0,image_1780764472561.jpg,kaggle,persistent-landing/unstructured/image/image_1780764472561.jpg,Image,2026-06-06 16:52:36.644648,1,"{""label"": ""sandstorm"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 8113, ""content_type"": ""image/jpg"", ""width"": 360, ""height"": 270, ""aspect_ratio"": 1.33, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""ce466219cc0a91a72cabdd02ea305a00""}",2026-06-06 16:52:36.644671,kaggle,2026-06-06T16:52:36.644690+00:00,s3://landing-zone/persistent-landing/unstructured/image/image_1780764472561.jpg,valid,landing_file_catalog_v1,data_engineering_team,bdm_project_team,public_image_metadata,no_direct_pii,course_project_retained_until_assessment_archive
1,image_1780764472606.jpg,kaggle,persistent-landing/unstructured/image/image_1780764472606.jpg,Image,2026-06-06 16:52:36.650545,1,"{""label"": ""sandstorm"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 39517, ""content_type"": ""image/jpg"", ""width"": 565, ""height"": 285, ""aspect_ratio"": 1.98, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""7337a063850d7ad3940074202796068d""}",2026-06-06 16:52:36.650620,kaggle,2026-06-06T16:52:36.650647+00:00,s3://landing-zone/persistent-landing/unstructured/image/image_1780764472606.jpg,valid,landing_file_catalog_v1,data_engineering_team,bdm_project_team,public_image_metadata,no_direct_pii,course_project_retained_until_assessment_archive
2,image_1780764472651.jpg,kaggle,persistent-landing/unstructured/image/image_1780764472651.jpg,Image,2026-06-06 16:52:36.655440,1,"{""label"": ""sandstorm"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 20671, ""content_type"": ""image/jpg"", ""width"": 282, ""height"": 148, ""aspect_ratio"": 1.91, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""608b2d018462b02892c92afb42137369""}",2026-06-06 16:52:36.655465,kaggle,2026-06-06T16:52:36.655489+00:00,s3://landing-zone/persistent-landing/unstructured/image/image_1780764472651.jpg,valid,landing_file_catalog_v1,data_engineering_team,bdm_project_team,public_image_metadata,no_direct_pii,course_project_retained_until_assessment_archive
3,image_1780764472706.jpg,kaggle,persistent-landing/unstructured/image/image_1780764472706.jpg,Image,2026-06-06 16:52:36.659108,1,"{""label"": ""sandstorm"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 24382, ""content_type"": ""image/jpg"", ""width"": 360, ""height"": 197, ""aspect_ratio"": 1.83, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""f6d85388e8189dd015a383541b45048d""}",2026-06-06 16:52:36.659130,kaggle,2026-06-06T16:52:36.659150+00:00,s3://landing-zone/persistent-landing/unstructured/image/image_1780764472706.jpg,valid,landing_file_catalog_v1,data_engineering_team,bdm_project_team,public_image_metadata,no_direct_pii,course_project_retained_until_assessment_archive
4,image_1780764472755.jpg,kaggle,persistent-landing/unstructured/image/image_1780764472755.jpg,Image,2026-06-06 16:52:36.662694,1,"{""label"": ""sandstorm"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 9405, ""content_type"": ""image/jpg"", ""width"": 419, ""height"": 240, ""aspect_ratio"": 1.75, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""d9121c373ea8be965726a09040ac18a1""}",2026-06-06 16:52:36.662713,kaggle,2026-06-06T16:52:36.662728+00:00,s3://landing-zone/persistent-landing/unstructured/image/image_1780764472755.jpg,valid,landing_file_catalog_v1,data_engineering_team,bdm_project_team,public_image_metadata,no_direct_pii,course_project_retained_until_assessment_archive
...,...,...